# DBSCAN — плотностной алгоритм кластеризации с выделением шума

## 1. Название и краткая информация о сдаваемом методе (1 абзац)

**DBSCAN** (читается «ди-би-скан», от англ. *Density-Based Spatial Clustering of Applications with Noise* — «пространственная кластеризация на основе плотности с выделением шума») — это алгоритм **кластеризации без учителя** (unsupervised learning, «обучение без учителя»: модели заранее не сообщают, к какому классу относится объект, она должна сама найти группы в данных). Идея метода очень наглядная: группой (кластером) считается **плотное скопление точек**, то есть такой участок пространства, где рядом друг с другом находится много объектов. Формально используются два параметра — **eps** (эпсилон, радиус соседства: на каком расстоянии точки считаются «соседями») и **min_samples** (минимальное число соседей, чтобы точку считать «ядровой», то есть центром плотного участка). Точки, попавшие в плотную область, объединяются в один кластер; точки, у которых соседей мало и которые не примыкают к плотной области, помечаются как **шум** (noise, метка `-1`). Главные преимущества DBSCAN: не нужно заранее задавать число кластеров (в отличие от K-means), метод умеет находить кластеры **произвольной формы** (не только шарообразные) и явно выделяет выбросы. Главное требование — признаки должны быть приведены к одному масштабу (стандартизация), иначе расстояние будет искажено признаком с большим разбросом значений.

## 2. Блок с используемыми библиотеками

Короткие пояснения к названиям:
- **numpy** (читается «нам-пай») — библиотека для численных операций с массивами.
- **pandas** (читается «панд-эс») — таблицы (DataFrame, «дата-фрейм») и удобная работа с данными.
- **matplotlib** — базовая библиотека графиков.
- **seaborn** (читается «си-борн») — красивые статистические графики поверх matplotlib (в том числе тепловая карта).
- **scikit-learn** (читается «сайкит-лёрн») — библиотека машинного обучения: готовые датасеты, модели и метрики.
- **DBSCAN** — сам алгоритм кластеризации из scikit-learn.
- **StandardScaler** — стандартизатор: приводит признаки к одному масштабу (среднее 0, разброс 1). Для DBSCAN это обязательный шаг, потому что метод работает с расстояниями между объектами.
- **NearestNeighbors** — помощник для поиска ближайших соседей; используется, чтобы подобрать параметр `eps` по графику k-расстояний.
- **PCA** (*Principal Component Analysis* — «метод главных компонент») — способ сжать много признаков в 2 для наглядной картинки.
- **silhouette_score, adjusted_rand_score** — метрики качества кластеризации (объяснены ниже).

In [ ]:
# Если вы запускаете в Google Colab, обычно всё уже установлено.
# Этот блок оставлен для надёжности: установит пакеты, если их вдруг нет.
# !pip -q install numpy pandas matplotlib seaborn scikit-learn

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

# Набор данных и вспомогательные инструменты
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA

# Сам алгоритм
from sklearn.cluster import DBSCAN

# Метрики качества кластеризации
from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    adjusted_rand_score,
    normalized_mutual_info_score,
    confusion_matrix,
)

# Для повторяемости результатов
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid")

## 3. Блок с описанием и демонстрацией (частичной распечаткой) используемого датасета

В качестве примера возьмём классический встроенный датасет **Iris** (в переводе — «Ирисы») из `scikit-learn`. Это набор из 150 цветков трёх видов ириса (*setosa*, *versicolor*, *virginica*), для каждого из которых измерены 4 числовых признака в сантиметрах:
- `sepal length` — длина чашелистика,
- `sepal width` — ширина чашелистика,
- `petal length` — длина лепестка,
- `petal width` — ширина лепестка.

Важное замечание: DBSCAN — метод **без учителя**, поэтому метки классов (вид цветка) он **не использует** при обучении. Мы будем использовать их только **в конце**, чтобы сравнить найденные кластеры с настоящими видами и оценить качество.

In [ ]:
data = load_iris()

X = pd.DataFrame(data.data, columns=data.feature_names)
y_true = pd.Series(data.target, name="target")          # настоящие метки видов: 0, 1, 2
target_names = list(data.target_names)                   # ['setosa', 'versicolor', 'virginica']

df = X.copy()
df["target"] = y_true

print("Размерность датасета (строки, столбцы):", df.shape)
print("Имена видов (классов):", target_names)
print("\nРаспределение настоящих классов (сколько цветков каждого вида):")
print(y_true.value_counts().sort_index())

print("\nПервые 5 строк датасета:")
display(df.head(5))

print("\nОписательная статистика по признакам:")
display(df.describe().T)

**Пояснение к таблицам:** каждая строка — один цветок, 4 числовых столбца — его геометрические признаки в см, а `target` — это правильный вид (0 = *setosa*, 1 = *versicolor*, 2 = *virginica*). Напомним ещё раз: `target` нужен только для финальной проверки качества кластеризации — в обучение DBSCAN он не подаётся.

## 4. Блок с предварительной обработкой датасета

Что делаем:
1. Проверяем **пропуски** (пустые значения) и **дубликаты** (повторяющиеся строки).
2. **Стандартизируем признаки** через `StandardScaler` — приводим каждый столбец к нулевому среднему и единичному разбросу. Для DBSCAN это **обязательный** шаг, потому что алгоритм опирается на расстояния между точками: если один признак измеряется в десятках, а другой — в единицах, расстояние «перекосится» в сторону крупного признака и плотностные группы исказятся.
3. В отличие от методов с учителем, здесь **не делим данные на train/test**: кластеризация ищет группы во всём наборе данных сразу.

In [ ]:
# 1) Пропуски
missing = df.isna().sum().sort_values(ascending=False)
print("Пропуски по каждому столбцу (если везде 0 — пропусков нет):")
print(missing)

# 2) Дубликаты
dup_count = df.duplicated().sum()
print("\nКоличество полностью совпадающих строк (дубликатов):", dup_count)

# 3) Стандартизация признаков (очень важна для DBSCAN)
scaler = StandardScaler()
X_scaled_array = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled_array, columns=X.columns, index=X.index)

print("\nРазмерность X после стандартизации:", X_scaled.shape)
print("\nПервые 5 строк после стандартизации (среднее ~ 0, разброс ~ 1):")
display(X_scaled.head(5))

print("\nСредние и стандартные отклонения после стандартизации (для проверки):")
display(pd.DataFrame({
    "среднее": X_scaled.mean().round(3),
    "ст. отклонение": X_scaled.std().round(3),
}))

## 5. Блок с тепловой картой

Тепловая карта (*heatmap* — «карта тепла») корреляций показывает, **насколько сильно связаны** друг с другом признаки. Значение корреляции меняется от -1 до 1:
- около 1 — сильная прямая связь (растёт один — растёт другой),
- около -1 — сильная обратная связь,
- около 0 — линейной связи почти нет.

Для кластеризации сильно скоррелированные признаки означают, что они несут похожую информацию. Это не мешает DBSCAN работать, но полезно понимать структуру данных.

In [ ]:
# Корреляции считаем по всему датасету (включая target — только для наглядности).
corr = df.corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr,
    annot=True,         # подписываем значения
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
)
plt.title("Тепловая карта корреляций признаков (и целевой переменной)")
plt.tight_layout()
plt.show()

print("Корреляции признаков с target (по убыванию модуля):")
print(
    corr["target"]
    .drop("target")
    .reindex(corr["target"].drop("target").abs().sort_values(ascending=False).index)
)

## 6. Блок с обучением модели

Для DBSCAN ключевых параметров два:
- **eps** (эпсилон) — радиус соседства. Две точки считаются соседями, если евклидово расстояние между ними меньше `eps`.
- **min_samples** — минимальное число точек в окрестности радиуса `eps`, чтобы точку считать «ядровой» (*core point*). Ядровые точки — это точки в плотных областях, вокруг которых и формируются кластеры.

**Типичное правило для min_samples:** берут `min_samples = 2 * число_признаков` (для Iris это 8) или не меньше `dim + 1`. Мы возьмём `min_samples = 5` — стандартное значение, хорошо работающее на небольших датасетах.

**Подбор eps** делаем по классическому приёму — **графику k-расстояний** (*k-distance plot*). Для каждой точки находим расстояние до её `k`-го ближайшего соседа (где `k = min_samples`), сортируем эти расстояния по возрастанию и ищем на графике **«изгиб» (колено, knee/elbow)** — точку, где кривая резко меняет крутизну. Значение расстояния в этом изгибе и берут как `eps`.

In [ ]:
# --- Шаг 1. Подбираем eps через график k-расстояний ---
min_samples = 5   # стандартное значение для небольших датасетов

# Для каждой точки ищем расстояния до min_samples ближайших соседей
nn = NearestNeighbors(n_neighbors=min_samples)
nn.fit(X_scaled)
distances, indices = nn.kneighbors(X_scaled)

# Берём расстояние до самого дальнего из этих соседей (k-го) и сортируем
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(9, 5))
plt.plot(k_distances)
plt.xlabel("Точки (отсортированы по k-расстоянию)")
plt.ylabel(f"Расстояние до {min_samples}-го ближайшего соседа")
plt.title("График k-расстояний для подбора параметра eps")
plt.grid(True)
plt.show()

print("Подсказка: смотрим, где кривая резко поднимается вверх (изгиб),")
print("и берём значение eps примерно в этой точке.")
print(f"\nМаксимальное k-расстояние: {k_distances.max():.3f}")
print(f"Медиана k-расстояний:       {np.median(k_distances):.3f}")

In [ ]:
# --- Шаг 2. Обучаем DBSCAN с подобранными параметрами ---
# По графику выше видно изгиб в районе 0.6–0.9. Возьмём eps = 0.8.
eps = 0.8

dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric="euclidean")
cluster_labels = dbscan.fit_predict(X_scaled)

# Метка -1 означает «шум» (noise) — точки, не попавшие ни в один кластер.
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = int(np.sum(cluster_labels == -1))

print(f"Параметры DBSCAN: eps = {eps}, min_samples = {min_samples}")
print(f"Найдено кластеров: {n_clusters}")
print(f"Объектов, помеченных как шум (метка -1): {n_noise} из {len(cluster_labels)}")

print("\nРаспределение объектов по найденным кластерам:")
print(pd.Series(cluster_labels, name="кластер").value_counts().sort_index())

**Пояснение к параметрам модели DBSCAN:**
- `eps` — радиус соседства. Меньше `eps` → больше мелких кластеров и больше шума. Больше `eps` → кластеры сливаются в один.
- `min_samples` — чем больше, тем «строже» определение плотной области и тем больше точек попадёт в шум.
- `metric="euclidean"` — евклидово (обычное) расстояние между точками.

Полезные атрибуты обученной модели:
- `labels_` — метки кластеров для каждой точки (`-1` означает шум).
- `core_sample_indices_` — индексы «ядровых» точек (центров плотных областей).

## 7. Блок с прогнозами модели

Так как DBSCAN — метод **без учителя**, «прогноз» здесь — это номер кластера, который модель присвоила каждому объекту (или метка `-1`, если объект отнесён к шуму). Оценим качество кластеризации сразу несколькими способами:

- **silhouette_score** (*силуэтная оценка*) — насколько точки внутри одного кластера плотные и насколько далеко от других кластеров. Значение от -1 до 1; ближе к 1 — лучше.
- **adjusted_rand_score (ARI, скорректированный индекс Рэнда)** — сравнивает найденные кластеры с настоящими метками. 1 — идеальное совпадение, 0 — как случайное угадывание.
- **normalized_mutual_info_score (NMI, нормированная взаимная информация)** — тоже сравнивает с истиной, значение от 0 до 1; чем больше — тем лучше.
- **confusion_matrix (таблица сопряжённости)** — наглядное сопоставление «истинный вид × номер кластера», чтобы увидеть, какой кластер какому виду соответствует.

In [ ]:
# Метрики, сравнивающие с настоящими метками (внешние метрики)
ari = adjusted_rand_score(y_true, cluster_labels)
nmi = normalized_mutual_info_score(y_true, cluster_labels)

print(f"Adjusted Rand Index (ARI):             {ari:.4f}")
print(f"Normalized Mutual Information (NMI):   {nmi:.4f}")

# Силуэтная оценка считается только по не-шумовым точкам и только если кластеров >= 2.
mask_not_noise = cluster_labels != -1
unique_non_noise = set(cluster_labels[mask_not_noise])
if len(unique_non_noise) >= 2:
    sil = silhouette_score(X_scaled[mask_not_noise], cluster_labels[mask_not_noise])
    print(f"Silhouette score (без учёта шума):     {sil:.4f}")
else:
    sil = None
    print("Silhouette score посчитать нельзя: найден <2 кластеров без шума.")

# Таблица сопряжённости «истинный вид × номер кластера»
crosstab = pd.crosstab(
    pd.Series(y_true.map(dict(enumerate(target_names))), name="Истинный вид"),
    pd.Series(cluster_labels, name="Кластер DBSCAN"),
)
print("\nТаблица сопряжённости (истинный вид × найденный кластер):")
display(crosstab)

# Частичная распечатка прогнозов
preview = pd.DataFrame({
    "истинный вид": y_true.map(dict(enumerate(target_names))).values,
    "номер кластера": cluster_labels,
    "это шум?": np.where(cluster_labels == -1, "да", "нет"),
})
# Покажем перемешанную выборку, чтобы в превью попали все виды
preview_sample = preview.sample(n=15, random_state=RANDOM_STATE).reset_index(drop=True)
print("\nПример прогнозов (15 случайных объектов):")
display(preview_sample)

**Как читать метрики:**
- **ARI ~ 0.5–0.6** — это нормальный результат на Iris: *setosa* отделяется от остальных двух видов идеально, а *versicolor* и *virginica* сильно пересекаются по признакам, поэтому DBSCAN часто объединяет их в один плотный кластер.
- **Silhouette ~ 0.5+** означает, что точки внутри кластера действительно плотные.
- В таблице сопряжённости видно, сколько цветков каждого вида попало в каждый найденный кластер (и сколько — в шум, метка `-1`).

## 8. Блок с графиками выходных результатов

Построим четыре графика:
1. **Кластеры на плоскости PCA** (метод главных компонент — способ сжать 4 признака в 2 для наглядности). Отдельно подсветим шумовые точки.
2. **Сравнение настоящих видов и найденных кластеров** на той же плоскости PCA.
3. **Силуэтный график** — для каждой точки показывает, насколько уверенно она попала в свой кластер.
4. **Чувствительность DBSCAN к параметру eps** — как меняется число кластеров, доля шума и ARI при разных значениях `eps`.

In [ ]:
# 8.1 Кластеры DBSCAN на плоскости PCA
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_.sum()

plt.figure(figsize=(9, 7))

# Рисуем не-шумовые точки разными цветами по номеру кластера
unique_labels = sorted(set(cluster_labels))
palette = sns.color_palette("tab10", n_colors=max(len(unique_labels), 3))

for i, lbl in enumerate(unique_labels):
    mask = cluster_labels == lbl
    if lbl == -1:
        plt.scatter(
            X_pca[mask, 0], X_pca[mask, 1],
            s=60, c="black", marker="x", linewidths=1.5,
            label="шум (-1)",
        )
    else:
        plt.scatter(
            X_pca[mask, 0], X_pca[mask, 1],
            s=50, color=palette[i % len(palette)], edgecolor="k",
            label=f"кластер {lbl}",
        )

plt.title(f"Кластеры DBSCAN на плоскости PCA (2 компоненты, eps={eps})")
plt.xlabel(f"Главная компонента 1 ({pca.explained_variance_ratio_[0]:.1%} дисперсии)")
plt.ylabel(f"Главная компонента 2 ({pca.explained_variance_ratio_[1]:.1%} дисперсии)")
plt.legend(loc="best")
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"Доля объяснённой дисперсии двумя компонентами PCA: {explained:.3f}")

In [ ]:
# 8.2 Сравнение бок-о-бок: настоящие виды и найденные кластеры
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Слева — настоящие метки видов
for i, name in enumerate(target_names):
    mask = y_true.values == i
    axes[0].scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        s=55, edgecolor="k", label=name,
    )
axes[0].set_title("Настоящие виды ирисов (истина)")
axes[0].set_xlabel("Главная компонента 1")
axes[0].set_ylabel("Главная компонента 2")
axes[0].legend()
axes[0].grid(True)

# Справа — кластеры DBSCAN
for i, lbl in enumerate(unique_labels):
    mask = cluster_labels == lbl
    if lbl == -1:
        axes[1].scatter(
            X_pca[mask, 0], X_pca[mask, 1],
            s=55, c="black", marker="x", linewidths=1.5,
            label="шум (-1)",
        )
    else:
        axes[1].scatter(
            X_pca[mask, 0], X_pca[mask, 1],
            s=55, color=palette[i % len(palette)], edgecolor="k",
            label=f"кластер {lbl}",
        )
axes[1].set_title(f"Кластеры DBSCAN (eps={eps}, min_samples={min_samples})")
axes[1].set_xlabel("Главная компонента 1")
axes[1].set_ylabel("Главная компонента 2")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# 8.3 Силуэтный график для не-шумовых точек
# Силуэтный коэффициент для каждой точки: 1 — идеально своя, 0 — на границе, -1 — скорее чужая.

if len(unique_non_noise) >= 2:
    X_ok = X_scaled[mask_not_noise]
    labels_ok = cluster_labels[mask_not_noise]
    silhouette_vals = silhouette_samples(X_ok, labels_ok)
    avg_silhouette = silhouette_vals.mean()

    plt.figure(figsize=(9, 6))
    y_lower = 10
    for i, lbl in enumerate(sorted(set(labels_ok))):
        vals = np.sort(silhouette_vals[labels_ok == lbl])
        size = len(vals)
        y_upper = y_lower + size
        plt.fill_betweenx(
            np.arange(y_lower, y_upper),
            0, vals,
            color=palette[i % len(palette)],
            alpha=0.8,
        )
        plt.text(-0.02, y_lower + 0.5 * size, f"кластер {lbl}", fontsize=10)
        y_lower = y_upper + 10

    plt.axvline(avg_silhouette, color="red", linestyle="--",
                label=f"среднее значение = {avg_silhouette:.3f}")
    plt.xlabel("Значение силуэтного коэффициента")
    plt.ylabel("Объекты (сгруппированы по кластерам)")
    plt.title("Силуэтный график кластеров DBSCAN (без учёта шума)")
    plt.legend(loc="lower right")
    plt.yticks([])
    plt.grid(True, axis="x")
    plt.tight_layout()
    plt.show()
else:
    print("Силуэтный график не построен: нужно минимум 2 кластера без шума.")

In [ ]:
# 8.4 Чувствительность DBSCAN к параметру eps
# Перебираем разные eps и смотрим, как меняются:
#   - число найденных кластеров,
#   - доля точек, отнесённых к шуму,
#   - метрика ARI (совпадение с настоящими видами).

eps_grid = np.round(np.arange(0.2, 1.81, 0.1), 2)
rows = []
for e in eps_grid:
    model_e = DBSCAN(eps=e, min_samples=min_samples).fit(X_scaled)
    labels_e = model_e.labels_
    n_c = len(set(labels_e)) - (1 if -1 in labels_e else 0)
    noise_frac = np.mean(labels_e == -1)
    ari_e = adjusted_rand_score(y_true, labels_e)
    rows.append({"eps": e, "кластеров": n_c, "доля шума": noise_frac, "ARI": ari_e})

sweep = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(sweep["eps"], sweep["кластеров"], marker="o")
axes[0].set_title("Число кластеров от eps")
axes[0].set_xlabel("eps")
axes[0].set_ylabel("Количество кластеров")
axes[0].grid(True)

axes[1].plot(sweep["eps"], sweep["доля шума"], marker="s", color="darkorange")
axes[1].set_title("Доля шумовых точек от eps")
axes[1].set_xlabel("eps")
axes[1].set_ylabel("Доля точек с меткой -1")
axes[1].grid(True)

axes[2].plot(sweep["eps"], sweep["ARI"], marker="^", color="green")
axes[2].axvline(eps, color="red", linestyle="--", label=f"выбранный eps = {eps}")
axes[2].set_title("ARI (совпадение с видами) от eps")
axes[2].set_xlabel("eps")
axes[2].set_ylabel("Adjusted Rand Index")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

print("Сводная таблица по перебранным значениям eps:")
display(sweep)

---

### Короткий вывод

Алгоритм **DBSCAN** применён к стандартизированному датасету Iris. Параметр `eps` подобран по графику k-расстояний (изгиб кривой), параметр `min_samples` взят стандартным (5). Модель без учителя сама нашла плотные группы в пространстве признаков и отдельно пометила выбросы (шум).

Графики показали:
- как выглядят найденные кластеры на плоскости PCA и как они соотносятся с настоящими видами ирисов,
- силуэтные коэффициенты (насколько уверенно каждая точка попала в свой кластер),
- чувствительность метода к параметру `eps` (при маленьком `eps` всё становится шумом, при большом — всё сливается в один кластер).

**Ключевая особенность DBSCAN:** метод не требует заранее задавать число кластеров, умеет находить группы произвольной формы и явно выделяет выбросы. Плата за это — нужно аккуратно подбирать параметры `eps` и `min_samples`, а также обязательно стандартизировать признаки перед запуском.